# 00. GK-2A v1 데이터 점검

**목적**: `data/gk2a_v1/` 에 모인 위성 SWR 데이터의 컬럼·시간 범위·사이트별 커버리지·status 의미를 단독 점검하고, 후속 노트북에서 사용할 **status 필터 정책**을 확정.

다른 데이터셋(solar_hourly, asos_hourly 등)과 섞지 않고 GK-2A v1만 단독 점검.

In [9]:
import sys
from pathlib import Path
import glob
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DATA_DIR = Path('../../data/gk2a_v1')
files = sorted(DATA_DIR.glob('*.csv'))
print(f'CSV 파일 수: {len(files)}')
print(f'첫: {files[0].name}, 끝: {files[-1].name}')

CSV 파일 수: 48
첫: 202201.csv, 끝: 202512.csv


## 1. 컬럼 목록 + 데이터 타입

In [10]:
sample = pd.read_csv(files[0], parse_dates=['datetime_utc', 'datetime_kst'], nrows=5)
print(f'컬럼 수: {len(sample.columns)}')
pd.DataFrame({
    'column': sample.columns,
    'dtype': [str(sample[c].dtype) for c in sample.columns],
    'sample': [sample[c].iloc[0] for c in sample.columns],
})

컬럼 수: 11


,column,dtype,sample
0,datetime_utc,datetime64[us],2022-01-01 00:00:00
1,datetime_kst,datetime64[us],2022-01-01 09:00:00
2,site,str,경상대
3,dsr,float64,240.4
4,dsr_dqf,float64,1.0
5,asr,float64,208.4
6,asr_dqf,float64,1.0
7,rsr,float64,54.5
8,rsr_dqf,float64,1.0
9,sw_dqf,int64,0


## 2. 컬럼별 의미 (참고)

| 컬럼 | 의미 | 단위 |
|------|------|------|
| `datetime_utc` | UTC 타임스탬프 (10분 간격) | datetime |
| `datetime_kst` | KST 타임스탬프 (UTC+9) | datetime |
| `site` | 발전소명 (좌표중복 collapse 후 11개) | str |
| `dsr` | Downward Shortwave Radiation (지표 도달 일사) | W/m² |
| `dsr_dqf` | DSR 품질 플래그 (1=good, 0=bad) | 0/1 |
| `asr` | Absorbed Shortwave Radiation | W/m² |
| `asr_dqf` | ASR 품질 플래그 | 0/1 |
| `rsr` | Reflected Shortwave Radiation | W/m² |
| `rsr_dqf` | RSR 품질 플래그 | 0/1 |
| `sw_dqf` | 결합 SW 플래그 (status 판정 기준) | 0/1 |
| `status` | 추출 결과 (`ok`/`nan_value`/`sw_dqf_reject`/`read_error_*`) | str |

**status 판정 로직** (`src/crawl/gk2a_v3/extract_nc.py`):
```
if dsr is None:               status = 'nan_value'
elif dsr_dqf != 1:            status = 'dsr_dqf_reject'
elif sw_dqf != 1:             status = 'sw_dqf_reject'
else:                         status = 'ok'
```

## 3. 시간 범위 (전체 로드)

In [11]:
df = pd.concat(
    [pd.read_csv(f, parse_dates=['datetime_utc', 'datetime_kst']) for f in files],
    ignore_index=True,
)
print(f'총 행: {len(df):,}')
print(f'KST 시작: {df.datetime_kst.min()}')
print(f'KST 끝  : {df.datetime_kst.max()}')
print(f'UTC 시작: {df.datetime_utc.min()}')
print(f'UTC 끝  : {df.datetime_utc.max()}')
print(f'간격(분): {df.sort_values("datetime_utc").datetime_utc.diff().dt.total_seconds().div(60).mode().iloc[0]:.0f}')

총 행: 2,287,538
KST 시작: 2022-01-01 09:00:00
KST 끝  : 2026-01-01 08:50:00
UTC 시작: 2022-01-01 00:00:00
UTC 끝  : 2025-12-31 23:50:00
간격(분): 0


## 4. 사이트별 시간 범위 + 행 수

In [12]:
site_summary = df.groupby('site').agg(
    rows=('datetime_kst', 'size'),
    kst_start=('datetime_kst', 'min'),
    kst_end=('datetime_kst', 'max'),
    ok=('status', lambda s: (s == 'ok').sum()),
    nan=('status', lambda s: (s == 'nan_value').sum()),
    reject=('status', lambda s: (s == 'sw_dqf_reject').sum()),
    err=('status', lambda s: s.str.startswith('read_error').sum()),
)
site_summary['ok_pct'] = (site_summary['ok'] / site_summary['rows'] * 100).round(2)
site_summary

,rows,kst_start,kst_end,ok,nan,reject,err,ok_pct
site,,,,,,,,
경상대,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,72958,119260,15739,1,35.08
고흥만수상,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,73265,119150,15542,1,35.23
광양항세방,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,73084,119203,15670,1,35.14
구미,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,72498,119504,15955,1,34.86
삼천포,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,73048,119218,15691,1,35.13
여수,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,73158,119173,15626,1,35.18
영동,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,72077,119746,16134,1,34.66
영흥,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,72003,119724,16230,1,34.62
예천,207958,2022-01-01 09:00:00,2026-01-01 08:50:00,72286,119623,16048,1,34.76


## 5. 컬럼별 결측·DQF 분포

In [13]:
value_cols = ['dsr', 'asr', 'rsr']
dqf_cols = ['dsr_dqf', 'asr_dqf', 'rsr_dqf', 'sw_dqf']

summary = pd.DataFrame({
    'non_null': [df[c].notna().sum() for c in value_cols + dqf_cols],
    'null': [df[c].isna().sum() for c in value_cols + dqf_cols],
    'null_pct': [round(df[c].isna().mean() * 100, 2) for c in value_cols + dqf_cols],
}, index=value_cols + dqf_cols)
summary

,non_null,null,null_pct
dsr,974443,1313095,57.4
asr,974414,1313124,57.4
rsr,974443,1313095,57.4
dsr_dqf,974443,1313095,57.4
asr_dqf,974443,1313095,57.4
rsr_dqf,974443,1313095,57.4
sw_dqf,2287527,11,0.0


In [14]:
print('--- status 전체 분포 ---')
print(df.status.value_counts())
print()
print('--- DQF 분포 (전체 행 기준) ---')
for c in dqf_cols:
    print(f'{c}: {df[c].value_counts(dropna=False).to_dict()}')

--- status 전체 분포 ---
status
nan_value             1313084
ok                     800226
sw_dqf_reject          174217
read_error_OSError         11
Name: count, dtype: int64

--- DQF 분포 (전체 행 기준) ---
dsr_dqf: {nan: 1313095, 1.0: 974443}
asr_dqf: {nan: 1313095, 1.0: 974407, 0.0: 36}
rsr_dqf: {nan: 1313095, 1.0: 974443}
sw_dqf: {0.0: 1487301, 1.0: 800226, nan: 11}


## 6. ok 행 기준 값 통계

In [15]:
ok = df[df.status == 'ok']
ok[value_cols].describe().round(2)

,dsr,asr,rsr
count,800226.00,800204.00,800226.00
mean,492.33,430.13,277.90
std,223.49,197.87,195.93
min,0.00,0.00,22.40
25%,330.20,285.70,140.00
50%,481.60,419.50,203.80
75%,655.30,573.90,359.00
max,1080.00,6543.50,1290.70


## 7. 연도·월별 row 수 (커버리지 매트릭스)

In [16]:
df['year'] = df.datetime_kst.dt.year
df['month'] = df.datetime_kst.dt.month
df.pivot_table(index='year', columns='month', values='datetime_kst', aggfunc='size', fill_value=0)

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2022,48158,44033,48741,47179,48642,47179,48741,48741,47179,48631,47179,48752
2023,48752,44033,48741,47179,48686,47179,48741,48741,47179,48752,47102,48752
2024,48752,45606,48741,47179,48675,47168,48741,48741,47179,48730,47091,48752
2025,48752,44033,48708,47179,48664,46651,47267,46398,44968,48642,45364,47971
2026,594,0,0,0,0,0,0,0,0,0,0,0


## 8. 시간대(KST) × status 매트릭스

**판독 기준**: 야간(19~05시)은 전부 `nan_value`여야 정상. 한낮(9~15시)은 거의 전부 `ok`. 일출/일몰 인접(6~8시, 16~18시)에 `sw_dqf_reject` 집중.

In [17]:
df['hour'] = df.datetime_kst.dt.hour
hour_status = df.groupby(['hour', 'status']).size().unstack(fill_value=0)
hour_status['total'] = hour_status.sum(axis=1)
for col in ['ok', 'sw_dqf_reject', 'nan_value']:
    if col in hour_status.columns:
        hour_status[f'{col}_pct'] = (hour_status[col] / hour_status['total'] * 100).round(1)
hour_status

status,nan_value,ok,read_error_OSError,sw_dqf_reject,total,ok_pct,sw_dqf_reject_pct,nan_value_pct
hour,,,,,,,,
0,90618,0,0,0,90618,0.0,0.0,100.0
1,95975,0,0,0,95975,0.0,0.0,100.0
2,96151,0,0,0,96151,0.0,0.0,100.0
3,96096,0,0,0,96096,0.0,0.0,100.0
4,96140,0,0,0,96140,0.0,0.0,100.0
5,96195,0,0,0,96195,0.0,0.0,100.0
6,78404,0,0,16988,95392,0.0,17.8,82.2
7,43288,23462,0,29390,96140,24.4,30.6,45.0
8,14708,56616,11,25245,96580,58.6,26.1,15.2


## 9. dawn/dusk 활용성 — `sw_dqf_reject` 행의 DSR은 살아있는가?

**판독 기준**: `sw_dqf_reject` = `dsr_dqf=1` 통과했지만 결합 `sw_dqf=0`. DSR 값 자체는 추출 시 살아있어야 함. 시간대별로 평균 DSR이 의미있는 수준이면 학습에 활용 가치 있음.

In [18]:
rej = df[df.status == 'sw_dqf_reject']
print(f'sw_dqf_reject 전체: {len(rej):,}')
print(f'  그중 DSR not null: {rej.dsr.notna().sum():,} ({rej.dsr.notna().mean()*100:.2f}%)')
print(f'  그중 DSR > 0     : {(rej.dsr > 0).sum():,} ({(rej.dsr > 0).mean()*100:.2f}%)')
print()
print('--- sw_dqf_reject 시간대별 DSR 통계 ---')
rej.groupby('hour').agg(
    n=('dsr', 'size'),
    dsr_zero=('dsr', lambda s: (s == 0).sum()),
    dsr_pos=('dsr', lambda s: (s > 0).sum()),
    dsr_min=('dsr', 'min'),
    dsr_mean=('dsr', 'mean'),
    dsr_max=('dsr', 'max'),
).round(1)

sw_dqf_reject 전체: 174,217
  그중 DSR not null: 174,217 (100.00%)
  그중 DSR > 0     : 169,717 (97.42%)

--- sw_dqf_reject 시간대별 DSR 통계 ---


,n,dsr_zero,dsr_pos,dsr_min,dsr_mean,dsr_max
hour,,,,,,
6,16988,652,16336,0.0,162.8,327.9
7,29390,1028,28362,0.0,202.1,394.9
8,25245,473,24772,0.0,217.1,399.5
9,15139,189,14950,0.0,250.8,400.1
10,118,0,118,82.9,258.4,324.5
15,14258,193,14065,0.0,259.9,401.1
16,24606,492,24114,0.0,218.8,390.4
17,25181,626,24555,0.0,211.8,401.5
18,23292,847,22445,0.0,179.2,384.2


## 10. 한낮(9~15시) DSR 결손 검증

**판독 기준**: 9~15시는 위성 정상 운영 시간. 이 구간 DSR null이 0%여야 위성 outage 무시 가능.

In [19]:
day_hours = list(range(7, 18))
day = df[df.hour.isin(day_hours)]
print('--- 시간대별 DSR null ---')
g = day.groupby('hour').agg(
    n=('dsr', 'size'),
    dsr_null=('dsr', lambda s: s.isna().sum()),
)
g['null_pct'] = (g['dsr_null'] / g['n'] * 100).round(2)
print(g)
print()
print('--- 7시·17시 DSR null의 월별 분포 (= 일출/일몰 경계의 천문학적 결손) ---')
edge = day[day.hour.isin([7, 17]) & day.dsr.isna()]
edge['ym'] = edge['datetime_kst'].dt.to_period('M').astype(str).str[-2:]
print(edge['ym'].value_counts().sort_index())

--- 시간대별 DSR null ---
          n  dsr_null  null_pct
hour                           
7     96140     43288     45.03
8     96580     14719     15.24
9     90156         0      0.00
10    95458         0      0.00
11    95942         0      0.00
12    95810         0      0.00
13    95843         0      0.00
14    95854         0      0.00
15    90827         0      0.00
16    95997     12595     13.12
17    95953     40054     41.74

--- 7시·17시 DSR null의 월별 분포 (= 일출/일몰 경계의 천문학적 결손) ---
ym
01    16360
02    12708
03     6735
04      350
09     3114
10    12177
11    15563
12    16335
Name: count, dtype: int64


→ **모델 함의**: 7시·17시 DSR null은 위성 장애가 아니라 *겨울에 해 안 떠있음*. 매년 11~2월에 집중되고 5~8월에는 0건. 이 행들은 PV 발전도 0이라 잃어도 문제 없음.

9~15시 DSR null = 0건이면 위성은 한낮 데이터를 100% 보존.

## 11. ~~status_v2 재정의~~ (deprecated, §13으로 대체)

초기 제안: `usable = dsr.notna()` / `unusable = dsr.isna()`

**기각 이유**: 같은 timestamp에서 사이트별로 일출/일몰 시각이 다름 (lat/lon 차이로 zenith가 다름). 위성 NaN 단일 기준은 이 spatial 패턴을 잃음.

→ §13~§15에서 **태양 천정각(solar zenith) 기반 status_v3**로 재정의.